In [ ]:
import pandas as pd
import numpy as np
import random
from typing import Set, Tuple, Dict, Any, List
from pathlib import Path
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
plt.style.use("./rw_visualization.mplstyle")

In [ ]:
data_path = Path("./signor")
data_oct_2018 = pd.read_csv(data_path / 'Oct2018_release.txt', sep='\t')
data_oct_2019 = pd.read_csv(data_path / 'Oct2019_release.txt', sep='\t')
data_oct_2020 = pd.read_csv(data_path / 'Oct2020_release.txt', sep='\t')
data_oct_2021 = pd.read_csv(data_path / 'Oct2021_release.txt', sep='\t')
data_oct_2022 = pd.read_csv(data_path / 'Oct2022_release.txt', sep='\t')
data_oct_2023 = pd.read_csv(data_path / 'Oct2023_release.txt', sep='\t')
data_oct_2024 = pd.read_csv(data_path / 'Oct2024_release.txt', sep='\t')
data_jul_2025 = pd.read_csv(data_path / 'Jul2025_release.txt', sep='\t')

# Use SIGNOR_ID

In [ ]:
oct_2018_edges = data_oct_2018.SIGNOR_ID
oct_2019_edges = data_oct_2019.SIGNOR_ID
oct_2020_edges = data_oct_2020.SIGNOR_ID
oct_2021_edges = data_oct_2021.SIGNOR_ID
oct_2022_edges = data_oct_2022.SIGNOR_ID
oct_2023_edges = data_oct_2023.SIGNOR_ID
oct_2024_edges = data_oct_2024.SIGNOR_ID
jul_2025_edges = data_jul_2025.SIGNOR_ID

# Set of unique edges
oct_2018_edge_set = set(oct_2018_edges)
oct_2019_edge_set = set(oct_2019_edges)
oct_2020_edge_set = set(oct_2020_edges)
oct_2021_edge_set = set(oct_2021_edges)
oct_2022_edge_set = set(oct_2022_edges)
oct_2023_edge_set = set(oct_2023_edges)
oct_2024_edge_set = set(oct_2024_edges)
jul_2025_edge_set = set(jul_2025_edges)

In [ ]:
removed_edges_2018_2019 = oct_2018_edge_set - oct_2018_edge_set.intersection(oct_2019_edge_set)
removed_edges_2019_2020 = oct_2019_edge_set - oct_2019_edge_set.intersection(oct_2020_edge_set)
removed_edges_2020_2021 = oct_2020_edge_set - oct_2020_edge_set.intersection(oct_2021_edge_set)
removed_edges_2021_2022 = oct_2021_edge_set - oct_2021_edge_set.intersection(oct_2022_edge_set)
removed_edges_2022_2023 = oct_2022_edge_set - oct_2022_edge_set.intersection(oct_2023_edge_set)
removed_edges_2023_2024 = oct_2023_edge_set - oct_2023_edge_set.intersection(oct_2024_edge_set)
removed_edges_2024_2025 = oct_2024_edge_set - oct_2024_edge_set.intersection(jul_2025_edge_set)

In [ ]:
plt.bar([0, 1, 2, 3, 4, 5], [len(removed_edges_2018_2019), len(removed_edges_2019_2020), len(removed_edges_2020_2021), len(removed_edges_2021_2022), len(removed_edges_2022_2023), len(removed_edges_2023_2024)])
plt.xticks([0, 1, 2, 3, 4, 5], ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024'])
# Rotate x-axis labels
plt.xticks(rotation=30)
plt.ylabel('Number of edges removed')
plt.show()

In [ ]:
# Is there any edges being remove twice? -> No
removed_edges_years = [removed_edges_2018_2019, removed_edges_2019_2020, removed_edges_2020_2021, removed_edges_2021_2022, removed_edges_2022_2023, removed_edges_2023_2024, removed_edges_2024_2025]
for i in range(len(removed_edges_years)):
    for j in range(i+1, len(removed_edges_years)):
        if removed_edges_years[i].intersection(removed_edges_years[j]):
            print(i, j, removed_edges_years[i].intersection(removed_edges_years[j]))

In [ ]:
def find_removed_edges_in_current_data(removed_edges: set, old_data: pd.DataFrame, new_data: pd.DataFrame):
    """
    Find removed edges (SIGNOR_ID) from old data (get source, target, interaction) and double check if they exist in new data
    """
    found_in_new_data = []
    for i, removed_edge in enumerate(removed_edges):
        # Step 1: Find the interaction details in the old data using SIGNOR_ID
        if not pd.isna(removed_edge):
            row = old_data[old_data['SIGNOR_ID'] == removed_edge]
        else:
            row = old_data[old_data['SIGNOR_ID'].isna()]
        # Extract interaction details from old_data
        source = row['ENTITYA']
        target = row['ENTITYB']
        interaction = row['EFFECT']

        # Step 2: Double check if the interaction exists in the new data
        matching_row = new_data[
            (new_data['ENTITYA'] == source.values[0]) & 
            (new_data['ENTITYB'] == target.values[0]) & 
            (new_data['EFFECT'] == interaction.values[0])
        ]
        if not matching_row.empty:
            found_in_new_data.append({
                'original_signor_id': removed_edge,
                'source': source.values[0],
                'target': target.values[0],
                'interaction': interaction.values[0],
                'new_signor_ids': matching_row['SIGNOR_ID'].tolist()[0]
            })
    return found_in_new_data

In [ ]:
found_in_new_data = find_removed_edges_in_current_data(removed_edges_2019_2020, data_oct_2019, data_oct_2020)
# There are plenty edges with different SIGNOR_ID from different years.

In [ ]:
removed_edges_list = [removed_edges_2018_2019, removed_edges_2019_2020, removed_edges_2020_2021, removed_edges_2021_2022, removed_edges_2022_2023, removed_edges_2023_2024, removed_edges_2024_2025]
old_data_list = [data_oct_2018, data_oct_2019, data_oct_2020, data_oct_2021, data_oct_2022, data_oct_2023, data_oct_2024]
new_data_list = [data_oct_2019, data_oct_2020, data_oct_2021, data_oct_2022, data_oct_2023, data_oct_2024, data_jul_2025]
num_existing_edges = []
for i, removed_edges in enumerate(removed_edges_list):
    found_in_new_data = find_removed_edges_in_current_data(removed_edges, old_data_list[i], new_data_list[i])
    num_existing_edges.append(len(found_in_new_data))

In [ ]:
plt.bar([0, 1, 2, 3, 4, 5], [len(removed_edges_2018_2019)-num_existing_edges[0], len(removed_edges_2019_2020)-num_existing_edges[1], len(removed_edges_2020_2021)-num_existing_edges[2], len(removed_edges_2021_2022)-num_existing_edges[3], len(removed_edges_2022_2023)-num_existing_edges[4], len(removed_edges_2023_2024)-num_existing_edges[5]])
plt.xticks([0, 1, 2, 3, 4, 5], ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024'])
# Rotate x-axis labels
plt.xticks(rotation=30)
plt.ylabel('Number of edges removed')
plt.show()

# Test by entity names - to be deleted

In [ ]:
# Save as csv
data_oct_2018.to_csv('data_oct_2018.csv', index=False)
data_oct_2019.to_csv('data_oct_2019.csv', index=False)

In [ ]:
# Get interactions
def create_interaction_key(df: pd.DataFrame, include_effect=True) -> List[Tuple[str, str, str]]:
    """
    Create a unique key for each interaction
    """
    key_columns = ['ENTITYA', 'ENTITYB']
    
    if include_effect:
        key_columns.append('EFFECT')

    edges = list(df[key_columns].itertuples(index=False, name=None))

    return edges

In [ ]:
include_effect = False
oct_2018_edges = create_interaction_key(data_oct_2018, include_effect=include_effect)
oct_2019_edges = create_interaction_key(data_oct_2019, include_effect=include_effect)
oct_2020_edges = create_interaction_key(data_oct_2020, include_effect=include_effect)
oct_2021_edges = create_interaction_key(data_oct_2021, include_effect=include_effect)
oct_2022_edges = create_interaction_key(data_oct_2022, include_effect=include_effect)
oct_2023_edges = create_interaction_key(data_oct_2023, include_effect=include_effect)
oct_2024_edges = create_interaction_key(data_oct_2024, include_effect=include_effect)
jul_2025_edges = create_interaction_key(data_jul_2025, include_effect=include_effect)

# Set of unique edges
oct_2018_edge_set = set(oct_2018_edges)
oct_2019_edge_set = set(oct_2019_edges)
oct_2020_edge_set = set(oct_2020_edges)
oct_2021_edge_set = set(oct_2021_edges)
oct_2022_edge_set = set(oct_2022_edges)
oct_2023_edge_set = set(oct_2023_edges)
oct_2024_edge_set = set(oct_2024_edges)
jul_2025_edge_set = set(jul_2025_edges)

In [ ]:
print('oct_2018 edges:', len(oct_2018_edges))
print('Unique oct_2018 edges:', len(oct_2018_edge_set))
print('oct_2019 edges:', len(oct_2019_edges))
print('Unique oct_2019 edges:', len(oct_2019_edge_set))
print('oct_2020 edges:', len(oct_2020_edges))
print('Unique oct_2020 edges:', len(oct_2020_edge_set))
print('oct_2021 edges:', len(oct_2021_edges))
print('Unique oct_2021 edges:', len(oct_2021_edge_set))
print('oct_2022 edges:', len(oct_2022_edges))
print('Unique oct_2022 edges:', len(oct_2022_edge_set))
print('oct_2023 edges:', len(oct_2023_edges))    
print('Unique oct_2023 edges:', len(oct_2023_edge_set))
print('oct_2024 edges:', len(oct_2024_edges))
print('Unique oct_2024 edges:', len(oct_2024_edge_set))
print('jul_2025 edges:', len(jul_2025_edges))
print('Unique jul_2025 edges:', len(jul_2025_edge_set))

In [ ]:
removed_edges_2018_2019 = oct_2018_edge_set - oct_2018_edge_set.intersection(oct_2019_edge_set)
removed_edges_2019_2020 = oct_2019_edge_set - oct_2019_edge_set.intersection(oct_2020_edge_set)
removed_edges_2020_2021 = oct_2020_edge_set - oct_2020_edge_set.intersection(oct_2021_edge_set)
removed_edges_2021_2022 = oct_2021_edge_set - oct_2021_edge_set.intersection(oct_2022_edge_set)
removed_edges_2022_2023 = oct_2022_edge_set - oct_2022_edge_set.intersection(oct_2023_edge_set)
removed_edges_2023_2024 = oct_2023_edge_set - oct_2023_edge_set.intersection(oct_2024_edge_set)
removed_edges_2024_2025 = oct_2024_edge_set - oct_2024_edge_set.intersection(jul_2025_edge_set)

In [ ]:
plt.bar([0, 1, 2, 3, 4, 5], [len(removed_edges_2018_2019), len(removed_edges_2019_2020), len(removed_edges_2020_2021), len(removed_edges_2021_2022), len(removed_edges_2022_2023), len(removed_edges_2023_2024)])
plt.xticks([0, 1, 2, 3, 4, 5], ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024'])
# Rotate x-axis labels
plt.xticks(rotation=30)
plt.ylabel('Number of edges removed')
plt.show()

In [ ]:
# Is there any edges being remove twice? -> No
removed_edges_years = [removed_edges_2018_2019, removed_edges_2019_2020, removed_edges_2020_2021, removed_edges_2021_2022, removed_edges_2022_2023, removed_edges_2023_2024, removed_edges_2024_2025]
for i in range(len(removed_edges_years)):
    for j in range(i+1, len(removed_edges_years)):
        if removed_edges_years[i].intersection(removed_edges_years[j]):
            print(i, j, removed_edges_years[i].intersection(removed_edges_years[j]))

In [ ]:
all_nodes_2025 = set()
for edges in jul_2025_edges:
    all_nodes_2025.add(edges[0])
    all_nodes_2025.add(edges[1])


In [ ]:
removed_nodes_2018_2019 = set()
for edges in removed_edges_2018_2019:
    if edges[0] not in all_nodes_2025:
        removed_nodes_2018_2019.add(edges[0])
    if edges[1] not in all_nodes_2025:
        removed_nodes_2018_2019.add(edges[1])

In [ ]:
removed_nodes_2018_2019